# 02 — Phase 2 pre-tests

Runs the full Phase 2 pre-test pipeline:
1. Pesaran CD (Python + R sanity check)
2. Panel unit roots — CIPS (primary, R via `plm`), IPS / Maddala-Wu (sensitivity)
3. Structural breaks — Bai-Perron (R `strucchange`) + Zivot-Andrews (R `urca`)
4. Cointegration — Pedroni (R `pco`), Westerlund (Python ECM-based, wild-cluster bootstrap), Engle-Granger residual ADF (R)

Pre-test sample: 1995–2020 (the largest balanced sub-window where SWIID `gini_disp` covers ≥70 countries). Headline panel remains 1990–2023.

In [ ]:
import os, subprocess
os.chdir('..')
RBIN = '/home/iliane/playground/granger/claude/tools/mamba_root/envs/r/bin'
os.environ['PATH'] = RBIN + ':' + os.environ['PATH']
os.environ['R_LIBS_USER'] = '/home/iliane/playground/granger/claude/tools/r_libs'
subprocess.run([f'{RBIN}/Rscript', 'R/pretests.R'], check=True)
subprocess.run([f'{RBIN}/Rscript', 'R/structural_breaks.R'], check=True)
subprocess.run([f'{RBIN}/Rscript', 'R/cointegration.R'], check=True)

In [ ]:
import sys; sys.path.insert(0, '.')
from src.tests.run_phase2 import main
main()

In [ ]:
import pandas as pd
for t in ['phase2_decision','pretests_cd_py','pretests_cd_R','pretests_cips','pretests_unitroot',
          'pretests_baiperron','pretests_zivotandrews','pretests_pedroni','pretests_westerlund',
          'pretests_eg_residual_adf']:
    p = f'outputs/tables/{t}.csv'
    print('---', t, '---')
    print(pd.read_csv(p).head(10).to_string(index=False))
    print()

**Phase 2 verdict**

- **Cross-sectional dependence**: strong on every variable (Pesaran CD p ≈ 0). Use second-gen tests (CIPS) for the headline.
- **Unit roots**: `v2x_corr` is **I(1)** (CIPS fails to reject in levels, rejects in differences; ZA fails to reject across 10/10 sample countries). `gini_disp` is **I(1)** under the trend specification (CIPS) and country-by-country ZA fails to reject in 7/10 sample countries.
- **Structural breaks**: cross-country mean Bai-Perron breaks at 1993–94, 2012–13, 2017 (no global 2008-2010 financial-crisis break).
- **Cointegration**: Pedroni `pco` rejects in only 1 of 7 statistics (the parametric panel-ADF, known to over-reject in finite samples); Pedroni group: 0/3 reject. EG residual ADF: 22/71 (31%) of countries cointegrated. Westerlund: 4/4 reject under wild-bootstrap p-values, but the implemented standardisation produces extreme stats — flagged for review.
- **Decision**: not cointegrated under the brief's 2-of-3 rule (1 of 3 test families reject). **Phase 3 model class: Panel VAR in first differences.**